# 03 — Tool Calling: the Raw Mechanics

Notebook 01 used `create_agent` to hide the tool-calling loop. This notebook **opens the box** and runs the loop by hand so you understand what the agent is doing for you.

**Provider:** `groq:qwen/qwen3-32b` — Groq exposes the model's `reasoning_content` in `additional_kwargs`, which lets you read the model's pre-tool-call thinking. Useful for debugging.

> Read [`03_tools.md`](./03_tools.md) alongside.


## Setup


In [20]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv()
model = init_chat_model("groq:qwen/qwen3-32b")


## Step 1 — Define a tool

Same `@tool` decorator pattern as notebook 01. The docstring + type hints generate the JSON schema the model sees.


In [21]:
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location.

    Args:
        location: The city and state, e.g. "Jamnagar, Gujarat"
    """
    return f"It is sunny in {location}."


### Inspect the schema the model will see

The decorator-generated schema is just a Pydantic model under the hood. Useful to peek at when debugging tool definitions.


In [22]:
print("Name:       ", get_weather.name)
print("Description:", get_weather.description)
print("Args schema:", get_weather.args_schema.model_json_schema())


Name:        get_weather
Description: Get the weather at a location.

    Args:
        location: The city and state, e.g. "Jamnagar, Gujarat"
Args schema: {'description': 'Get the weather at a location.\n\nArgs:\n    location: The city and state, e.g. "Jamnagar, Gujarat"', 'properties': {'location': {'title': 'Location', 'type': 'string'}}, 'required': ['location'], 'title': 'get_weather', 'type': 'object'}


## Step 2 — Bind the tool to the model

`bind_tools([...])` returns a new model that sends the tool schemas with every request. The model is now **aware** of the tool, but cannot run it — only the application code can.

When the model decides to use the tool, it returns an `AIMessage` with an empty `content` and a populated `tool_calls` list.


In [23]:
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What's the weather like in Jamnagar?")

print("content:", repr(response.content))
print("\ntool_calls:")
for call in response.tool_calls:
    print(f"  name: {call['name']}")
    print(f"  args: {call['args']}")
    print(f"  id:   {call['id']}")


content: ''

tool_calls:
  name: get_weather
  args: {'location': 'Jamnagar, Gujarat'}
  id:   wtp93ngcr


## Step 3 — The manual tool-execution loop

This is what `create_agent` does internally. Three steps:

1. **Model produces tool calls.** We append the resulting `AIMessage` to the message list.
2. **App executes each tool.** `get_weather.invoke(call)` runs the function and returns a `ToolMessage` with the same `tool_call_id` as the call — that link is how the model matches the result back to its request.
3. **Model sees the result and produces the final answer.** We re-invoke with the extended history.

In a real loop, step 3 might *itself* return more tool calls, so we'd repeat. Here we know one tool call is enough.


In [24]:
# Step 1: model decides to call the tool
messages = [{"role": "user", "content": "What's the weather in Jamnagar?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: app runs the tool, appends ToolMessage(s)
for call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(call)  # returns a ToolMessage
    messages.append(tool_result)

# Step 3: model produces the final answer using the tool result
final = model_with_tools.invoke(messages)
messages.append(final)

print(final.content)


The weather in Jamnagar, Gujarat is currently sunny ☀️. Let me know if you'd like additional details!


### Inspect the full message history

The list captures the full loop — same shape as `response['messages']` from `create_agent` in notebook 01. The two approaches produce the same artifact; `create_agent` just writes the loop for you.


In [25]:
for i, m in enumerate(messages):
    kind = m["role"] if isinstance(m, dict) else type(m).__name__
    content = m["content"] if isinstance(m, dict) else m.content
    print(f"[{i}] {kind}: {content!r}")
    if not isinstance(m, dict) and getattr(m, 'tool_calls', None):
        print(f"    tool_calls: {m.tool_calls}")


[0] user: "What's the weather in Jamnagar?"
[1] AIMessage: ''
    tool_calls: [{'name': 'get_weather', 'args': {'location': 'Jamnagar, Gujarat'}, 'id': 't0kbtcc79', 'type': 'tool_call'}]
[2] ToolMessage: 'It is sunny in Jamnagar, Gujarat.'
[3] AIMessage: "The weather in Jamnagar, Gujarat is currently sunny ☀️. Let me know if you'd like additional details!"


## Step 4 — Multiple tools, model picks the right one

With more than one tool bound, the model chooses which to call (or which to call **in parallel** — many providers support multiple tool calls in a single `AIMessage`).


In [26]:
@tool
def get_time(timezone: str) -> str:
    """Get the current local time for a given timezone.

    Args:
        timezone: An IANA timezone name, e.g. "Asia/Kolkata" or "America/New_York"
    """
    from datetime import datetime
    from zoneinfo import ZoneInfo
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")


multi_tool_model = model.bind_tools([get_weather, get_time])

response = multi_tool_model.invoke(
    "What is the weather in Jamnagar AND the current time in Asia/Kolkata?"
)

for call in response.tool_calls:
    print(f"{call['name']}({call['args']})")


get_weather({'location': 'Jamnagar, Gujarat'})
get_time({'timezone': 'Asia/Kolkata'})


## When to drop down to the manual loop

Prefer `create_agent` (notebook 01) for everything by default. Switch to the manual loop when you need:

- **Custom retry / fallback** between calls (e.g. fall back to a cheaper model on tool errors).
- **Approval gates** — show the human the tool call before executing it.
- **Programmatic intervention** based on intermediate state.
- **Debugging** — full visibility of every step.

**Next**: [`04_messages.ipynb`](./04_messages.ipynb) — the message schema in detail, including how `tool_call_id` actually links things.
